In [40]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.types import interrupt, Command
import os
from dotenv import load_dotenv

load_dotenv()

llm = AzureChatOpenAI(
        azure_endpoint=os.environ.get("AZURE_ENDPOINT"),
        azure_deployment="sample-gpt-4o-deployment",
        api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
        api_version="2024-12-01-preview",
        temperature=0.5,
        top_p=1.0,
        max_tokens=4096,
    )

class CodingAssistantState(TypedDict):
    task: str       # The coding task description
    code: str       # Generated Python code
    tests: str      # Generated unit tests

# Define prompts
code_prompt = ChatPromptTemplate.from_template(
    "Generate Python code for the following task. Only provide the code, no explanations:\n\n{task}"
)
test_prompt = ChatPromptTemplate.from_template(
    "Write comprehensive unit tests for this Python code:\n\n{code}"
)

# Create chains
code_chain = code_prompt | llm | StrOutputParser()
test_chain = test_prompt | llm | StrOutputParser()


def generate_code(state) -> Command[Literal["Take Human Review"]]:
    """Node 1: Generate Python code using LLM"""
    code = code_chain.invoke({"task": state["task"]})
    print("[NODE] generate_code - Code generated successfully!")
    return Command(goto="Take Human Review", update={"code": code})


def human_review(state) -> Command[Literal["Create Unit Tests"]]:
    """Node 2: Pause for human review using interrupt()"""
    print("[NODE] human_review - GRAPH IS PAUSED ⏸️ - Waiting for human input...")
    
    # INTERRUPT: This pauses execution and waits for human input!
    human_response = interrupt({
        "message": "Please review the generated code below.",
        "code": state["code"],
        "instructions": """
                            You can respond with either: {
                                                            'action': 'approve'
                                                        } 
                            or  {
                                    'action': 'edit',
                                    'edited_code': '<<your edited code>>'
                                }
                        """
    })
    
    print(f"[NODE] Take Human Review - Received response: {human_response}")
    
    # Process human response
    if human_response.get("action") == "edit" and "edited_code" in human_response:
        print("[NODE] Take Human Review - Using edited code from human")
        return Command(goto="Create Unit Tests", update={"code": human_response["edited_code"]})
    else:
        print("[NODE] Take Human Review - Code approved, proceeding to tests")
        return Command(goto="Create Unit Tests")


def create_tests(state) -> Command[Literal["__end__"]]:
    """Node 3: Generate unit tests for the code"""
    tests = test_chain.invoke({"code": state["code"]})
    print("[NODE] Create Unit Tests - Tests generated successfully!")
    return Command(goto=END, update={"tests": tests})

In [41]:
graph = StateGraph(CodingAssistantState)

# Add nodes
graph.add_node("Generate The Code", generate_code)
graph.add_node("Take Human Review", human_review)
graph.add_node("Create Unit Tests", create_tests)

# Set entry point
graph.set_entry_point("Generate The Code")

# IMPORTANT: MemorySaver is required for interrupt() to work!
# It saves the graph state when paused so it can resume later
workflow = graph.compile(checkpointer=MemorySaver())

# Save the visual graph to a PNG file
with open("15_graph.png", "wb") as f:
    f.write(workflow.get_graph().draw_mermaid_png())

thread_config = {"configurable": {"thread_id": "1"}}

result = workflow.invoke({
                            "task": "Write a Python function that takes a list of integers and returns the sum of all even numbers in the list.",
                            "code": "",
                            "tests": ""
                        }, config=thread_config)

[NODE] generate_code - Code generated successfully!
[NODE] human_review - GRAPH IS PAUSED ⏸️ - Waiting for human input...


In [42]:
print("=" * 60)
print("INTERRUPT STATUS")
print("=" * 60)

if "__interrupt__" in result:
    interrupt_info = result["__interrupt__"][0].value
    
    print("\n" + "-" * 60)
    print("MESSAGE:", interrupt_info.get("message", ""))
    print("\n" + "-" * 60)
    print("GENERATED CODE:")
    print("\n" + interrupt_info.get("code", ""))
    print("\n" + "-" * 60)
    print("INSTRUCTIONS:", interrupt_info.get("instructions", ""))
    print("\n" + "=" * 60)
    print("\n>>> Run the NEXT CELL to approve or edit the code <<<")
else:
    print("\n✓ No interrupt - workflow completed or not started")
    print(f"\nResult: {result}")

INTERRUPT STATUS

------------------------------------------------------------
MESSAGE: Please review the generated code below.

------------------------------------------------------------
GENERATED CODE:

```python
def sum_of_evens(numbers):
    return sum(num for num in numbers if num % 2 == 0)
```

------------------------------------------------------------
INSTRUCTIONS: 
                            You can respond with either: {
                                                            'action': 'approve'
                                                        } 
                            or  {
                                    'action': 'edit',
                                    'edited_code': '<<your edited code>>'
                                }
                        


>>> Run the NEXT CELL to approve or edit the code <<<


In [43]:
# # Resume with approval
# human_response = {"action": "approve"}

# print("=" * 60)
# print(f"\nSending response: {human_response}")
# print("=" * 60)

# # Resume the workflow with Command(resume=...)
# final_result = workflow.invoke(
#     Command(resume=human_response),
#     config=thread_config
# )

In [44]:
# Resume with Edited code
human_response = {
                    "action": "edit",
                    "edited_code": """def sum_of_even_numbers(numbers):
                    \"\"\"Returns the sum of all even numbers in the given list of integers.\"\"\"
                    if not isinstance(numbers, list):
                        raise TypeError("Input must be a list")
                    return sum(x for x in numbers if x % 2 == 0)"""
                }

print("=" * 60)
print(f"\nSending response: {human_response}")
print("=" * 60)

# Resume the workflow with Command(resume=...)
final_result = workflow.invoke(
    Command(resume=human_response),
    config=thread_config
)


Sending response: {'action': 'edit', 'edited_code': 'def sum_of_even_numbers(numbers):\n                    """Returns the sum of all even numbers in the given list of integers."""\n                    if not isinstance(numbers, list):\n                        raise TypeError("Input must be a list")\n                    return sum(x for x in numbers if x % 2 == 0)'}
[NODE] human_review - GRAPH IS PAUSED ⏸️ - Waiting for human input...
[NODE] Take Human Review - Received response: {'action': 'edit', 'edited_code': 'def sum_of_even_numbers(numbers):\n                    """Returns the sum of all even numbers in the given list of integers."""\n                    if not isinstance(numbers, list):\n                        raise TypeError("Input must be a list")\n                    return sum(x for x in numbers if x % 2 == 0)'}
[NODE] Take Human Review - Using edited code from human
[NODE] Create Unit Tests - Tests generated successfully!


In [45]:
final_code = final_result.get("code", "")
print(final_code)

def sum_of_even_numbers(numbers):
                    """Returns the sum of all even numbers in the given list of integers."""
                    if not isinstance(numbers, list):
                        raise TypeError("Input must be a list")
                    return sum(x for x in numbers if x % 2 == 0)


In [46]:
final_tests = final_result.get("tests", "")
print(final_tests)

Here are comprehensive unit tests for the `sum_of_even_numbers` function using the `unittest` framework in Python:

```python
import unittest

# The function to be tested
def sum_of_even_numbers(numbers):
    """Returns the sum of all even numbers in the given list of integers."""
    if not isinstance(numbers, list):
        raise TypeError("Input must be a list")
    return sum(x for x in numbers if x % 2 == 0)

# Unit test class
class TestSumOfEvenNumbers(unittest.TestCase):
    def test_all_even_numbers(self):
        """Test when the list contains only even numbers."""
        self.assertEqual(sum_of_even_numbers([2, 4, 6, 8]), 20)

    def test_all_odd_numbers(self):
        """Test when the list contains only odd numbers."""
        self.assertEqual(sum_of_even_numbers([1, 3, 5, 7]), 0)

    def test_mixed_numbers(self):
        """Test when the list contains both even and odd numbers."""
        self.assertEqual(sum_of_even_numbers([1, 2, 3, 4, 5, 6]), 12)

    def test_empty_l